In [1]:
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
import sympy


# =====================================================================
# 1. GRPO Objective & Unbiased KL Functions (Equations 1, 2, 3)
# =====================================================================

def compute_group_advantages(rewards: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """
    Computes group-relative advantages across G sampled completions for a prompt.
    rewards shape: [G]
    returns shape: [G]
    """
    mean_r = torch.mean(rewards)
    std_r = torch.std(rewards, unbiased=False)
    
    # Handle zero variance edge-case
    if std_r < eps:
        return torch.zeros_like(rewards)
        
    advantages = (rewards - mean_r) / (std_r + eps)
    return advantages


def compute_unbiased_kl(log_p_curr: torch.Tensor, log_p_ref: torch.Tensor) -> torch.Tensor:
    """
    Schulman's unbiased token-level KL estimator (Equation 2 in paper):
    KL = (pi_ref / pi_theta) - log(pi_ref / pi_theta) - 1
    """
    # delta = log(pi_ref) - log(pi_theta)
    log_ratio = log_p_ref - log_p_curr
    kl = torch.exp(log_ratio) - log_ratio - 1.0
    return kl


def grpo_loss_function(
    log_probs_curr: torch.Tensor,  # [G, T] - current policy log-probs
    log_probs_old: torch.Tensor,   # [G, T] - rollout policy log-probs (detached)
    log_probs_ref: torch.Tensor,   # [G, T] - reference policy log-probs (detached)
    advantages: torch.Tensor,      # [G]    - scalar advantages
    clip_eps: float = 0.2,         # Paper GRPO clip ratio
    beta_kl: float = 0.001          # Paper KL coefficient
) -> torch.Tensor:
    """
    Computes GRPO Loss (Equation 1 in paper).
    """
    group_size, seq_len = log_probs_curr.shape
    adv_expanded = advantages.unsqueeze(-1) # [G, 1] broadcast across sequence length
    
    # 1. Compute probability ratio: r_t = exp(log_p_curr - log_p_old)
    ratios = torch.exp(log_probs_curr - log_probs_old) # [G, T]
    
    # 2. PPO-style Clipped Surrogate Objective
    surr1 = ratios * adv_expanded
    surr2 = torch.clamp(ratios, 1.0 - clip_eps, 1.0 + clip_eps) * adv_expanded
    policy_surrogate = torch.min(surr1, surr2) # Objective to maximize
    
    # 3. Unbiased Token-Level KL Penalty
    kl_penalty = compute_unbiased_kl(log_probs_curr, log_probs_ref) # [G, T]
    
    # 4. Total Loss per token: - (Policy_Surrogate - beta * KL)
    token_loss = -(policy_surrogate - beta_kl * kl_penalty)
    
    # Mean loss across group completions and sequence length
    return token_loss.mean()


# =====================================================================
# 2. Deterministic Rule-Based Reward Engine (Equation 4)
# =====================================================================

class DeepSeekR1ZeroRewardEngine:
    """
    Rule-Based Reward Engine for R1-Zero math tasks (Section 2.2).
    Evaluates:
      1. R_format (+0.1) if structural <think>...</think><answer>...</answer> tags exist.
      2. R_acc (+1.0) if SymPy evaluates extracted answer as equal to target.
    """
    def __init__(self, format_reward_val: float = 0.1):
        self.fmt_val = format_reward_val
        # Match <think>...</think> and <answer>...</answer>
        self.format_regex = re.compile(
            r"^\s*<think>(.*?)</think>\s*<answer>(.*?)</answer>\s*$", 
            re.DOTALL
        )

    def evaluate_math_answer(self, pred_str: str, target_str: str) -> bool:
        """
        Uses SymPy to check mathematical equivalence between prediction and ground truth.
        """
        try:
            pred_expr = sympy.parse_expr(pred_str.strip())
            target_expr = sympy.parse_expr(target_str.strip())
            # Check if symbolic difference simplifies to zero
            return sympy.simplify(pred_expr - target_expr) == 0
        except Exception:
            # Fallback to direct string matching if SymPy parsing fails
            return pred_str.strip() == target_str.strip()

    def compute_reward(self, completion_text: str, ground_truth: str) -> float:
        r_format = 0.0
        r_acc = 0.0
        
        match = self.format_regex.match(completion_text)
        if match:
            r_format = self.fmt_val
            extracted_answer = match.group(2).strip()
            
            # Evaluate Accuracy via SymPy
            if self.evaluate_math_answer(extracted_answer, ground_truth):
                r_acc = 1.0
        
        total_reward = r_acc + r_format
        return total_reward


# =====================================================================
# 3. Verification Execution
# =====================================================================

if __name__ == "__main__":
    torch.manual_seed(42)
    
    # --- Test 1: GRPO Advantage & Loss Calculation ---
    group_size = 4
    seq_len = 5
    
    # Mock rewards for a group of 4 completions: [Wrong, Correct+Fmt, Wrong, Fmt Only]
    raw_rewards = torch.tensor([0.0, 1.1, 0.0, 0.1])
    advantages = compute_group_advantages(raw_rewards)
    
    # Mock log-probs
    log_probs_curr = torch.randn(group_size, seq_len, requires_grad=True)
    log_probs_old = log_probs_curr.detach() - 0.05
    log_probs_ref = log_probs_curr.detach() - 0.1
    
    loss = grpo_loss_function(log_probs_curr, log_probs_old, log_probs_ref, advantages)
    loss.backward()
    
    print("=== Module 1.2 Mechanics Verification ===")
    print("Raw Rewards:         ", raw_rewards.tolist())
    print("Calculated Advantages:", [round(a, 3) for a in advantages.tolist()])
    print(f"GRPO Loss Value:      {loss.item():.6f}")
    print(f"Gradients Computed:   {log_probs_curr.grad is not None and not torch.isnan(log_probs_curr.grad).any()}\n")
    
    # --- Test 2: Rule-Based Reward Engine ---
    reward_engine = DeepSeekR1ZeroRewardEngine()
    
    sample_1 = "<think> 12*4=48, 48-15=33 </think> <answer> 33 </answer>" # Correct + Format
    sample_2 = "<think> 12*4=48 </think> <answer> 30 </answer>"          # Wrong + Format
    sample_3 = "The answer is 33"                                         # Correct + No Format
    
    ground_truth = "33"
    
    print("Reward Engine Evaluation:")
    print(f"  Sample 1 (Valid CoT + Ans): Reward = {reward_engine.compute_reward(sample_1, ground_truth)} (Expected: 1.1)")
    print(f"  Sample 2 (Wrong Math):     Reward = {reward_engine.compute_reward(sample_2, ground_truth)} (Expected: 0.1)")
    print(f"  Sample 3 (No Tags):        Reward = {reward_engine.compute_reward(sample_3, ground_truth)} (Expected: 0.0)")

=== Module 1.2 Mechanics Verification ===
Raw Rewards:          [0.0, 1.100000023841858, 0.0, 0.10000000149011612]
Calculated Advantages: [-0.647, 1.725, -0.647, -0.431]
GRPO Loss Value:      0.000005
Gradients Computed:   True

Reward Engine Evaluation:
  Sample 1 (Valid CoT + Ans): Reward = 1.1 (Expected: 1.1)
  Sample 2 (Wrong Math):     Reward = 0.1 (Expected: 0.1)
  Sample 3 (No Tags):        Reward = 0.0 (Expected: 0.0)


In [2]:
import math
import re
import sympy
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

# =====================================================================
# 1. Custom R1 Tokenizer & Synthetic Arithmetic Task
# =====================================================================

PAD_TOKEN, UNK_TOKEN = "<pad>", "<unk>"
THINK_START, THINK_END = "<think>", "</think>"
ANS_START, ANS_END = "<answer>", "</answer>"

class R1Tokenizer:
    def __init__(self):
        special_tokens = [PAD_TOKEN, UNK_TOKEN, THINK_START, THINK_END, ANS_START, ANS_END]
        math_chars = [str(i) for i in range(10)] + ["+", "-", "*", "=", " "]
        self.vocab = special_tokens + math_chars
        self.token2id = {token: idx for idx, token in enumerate(self.vocab)}
        self.id2token = {idx: token for idx, token in enumerate(self.vocab)}
        self.pad_id = self.token2id[PAD_TOKEN]

    @property
    def vocab_size(self):
        return len(self.vocab)

    def encode(self, text: str) -> list[int]:
        tokens, i = [], 0
        while i < len(text):
            matched = False
            for st in [THINK_START, THINK_END, ANS_START, ANS_END]:
                if text[i:].startswith(st):
                    tokens.append(self.token2id[st])
                    i += len(st)
                    matched = True
                    break
            if not matched:
                tokens.append(self.token2id.get(text[i], self.token2id[UNK_TOKEN]))
                i += 1
        return tokens

    def decode(self, token_ids: list[int]) -> str:
        return "".join([self.id2token.get(idx, UNK_TOKEN) for idx in token_ids if idx != self.pad_id])


# =====================================================================
# 2. Causal Transformer Architecture
# =====================================================================

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
        mask = torch.full((max_seq_len, max_seq_len), float("-inf"))
        self.register_buffer("causal_mask", torch.triu(mask, diagonal=1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        q, k, v = self.qkv_proj(x).chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores + self.causal_mask[:T, :T]
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)


class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, max_seq_len)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class CausalLLM(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 64, n_heads: int = 4, n_layers: int = 3, max_seq_len: int = 128):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, n_heads, max_seq_len) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        B, T = input_ids.shape
        pos = torch.arange(0, T, device=input_ids.device).unsqueeze(0)
        x = self.token_emb(input_ids) + self.pos_emb(pos)
        for block in self.blocks:
            x = block(x)
        return self.lm_head(self.ln_f(x))


# =====================================================================
# 3. Rule-Based Reward Engine & GRPO Trainer
# =====================================================================

class R1ZeroRewardEngine:
    def __init__(self):
        self.format_regex = re.compile(r"^\s*<think>(.*?)</think>\s*<answer>(.*?)</answer>\s*$", re.DOTALL)

    def evaluate(self, completion_text: str, ground_truth: str) -> float:
        match = self.format_regex.match(completion_text)
        if not match:
            return 0.0
        
        r_format = 0.1
        extracted_ans = match.group(2).strip()
        
        try:
            r_acc = 1.0 if sympy.simplify(sympy.parse_expr(extracted_ans) - sympy.parse_expr(ground_truth)) == 0 else 0.0
        except Exception:
            r_acc = 1.0 if extracted_ans == ground_truth.strip() else 0.0
            
        return r_acc + r_format


class FullGRPOTrainer:
    def __init__(
        self, 
        policy: nn.Module, 
        ref_policy: nn.Module, 
        tokenizer: R1Tokenizer,
        group_size: int = 4, 
        lr: float = 3e-4,
        beta_kl: float = 0.001,
        ref_update_steps: int = 5  # Scaled down from 400 for local demonstration
    ):
        self.policy = policy
        self.ref_policy = ref_policy
        self.tokenizer = tokenizer
        self.group_size = group_size
        self.beta_kl = beta_kl
        self.ref_update_steps = ref_update_steps
        self.optimizer = torch.optim.AdamW(self.policy.parameters(), lr=lr)
        self.reward_engine = R1ZeroRewardEngine()

    def rollout_group(self, prompt_text: str, max_gen_len: int = 40):
        """Samples G outputs from the rollout policy at temperature 1.0 and records log_p_old."""
        self.policy.eval()
        prompt_ids = torch.tensor([self.tokenizer.encode(prompt_text)], dtype=torch.long)
        prompt_len = prompt_ids.shape[1]
        
        input_ids = prompt_ids.repeat(self.group_size, 1)
        sampled_log_probs = []

        with torch.no_grad():
            for _ in range(max_gen_len):
                logits = self.policy(input_ids)[:, -1, :] # Temperature = 1.0
                probs = F.softmax(logits, dim=-1)
                log_probs = F.log_softmax(logits, dim=-1)
                
                next_tokens = torch.multinomial(probs, num_samples=1)
                token_log_p = log_probs.gather(dim=-1, index=next_tokens)
                
                sampled_log_probs.append(token_log_p)
                input_ids = torch.cat([input_ids, next_tokens], dim=1)

        completions = input_ids[:, prompt_len:]
        old_log_probs = torch.cat(sampled_log_probs, dim=1) # [G, gen_len]
        return input_ids, completions, old_log_probs, prompt_len

    def step(self, prompt_text: str, ground_truth: str, step_idx: int):
        # 1. Periodic Reference Policy Update (pi_ref <- pi_theta)
        if step_idx > 0 and step_idx % self.ref_update_steps == 0:
            print(f"  [Pipeline] Step {step_idx}: Updating Reference Policy (pi_ref <- pi_theta)...")
            self.ref_policy.load_state_dict(self.policy.state_dict())

        # 2. Rollout G Completions
        full_seqs, completions, old_log_probs, prompt_len = self.rollout_group(prompt_text)

        # 3. Evaluate Rewards & Compute Group Advantages
        raw_rewards = []
        for g in range(self.group_size):
            comp_text = self.tokenizer.decode(completions[g].tolist())
            r = self.reward_engine.evaluate(comp_text, ground_truth)
            raw_rewards.append(r)
            
        rewards_t = torch.tensor(raw_rewards, dtype=torch.float32)
        
        # Advantage Normalization across Group
        std_r = torch.std(rewards_t, unbiased=False)
        advantages = (rewards_t - torch.mean(rewards_t)) / (std_r + 1e-8) if std_r > 1e-8 else torch.zeros_like(rewards_t)

        # 4. Current Policy Pass (Active Autograd)
        self.policy.train()
        logits_curr = self.policy(full_seqs)[:, prompt_len-1:-1, :]
        log_p_curr_all = F.log_softmax(logits_curr, dim=-1)
        
        targets = completions.unsqueeze(-1)
        curr_log_probs = log_p_curr_all.gather(dim=-1, index=targets).squeeze(-1)

        # 5. Reference Policy Pass (Frozen)
        with torch.no_grad():
            logits_ref = self.ref_policy(full_seqs)[:, prompt_len-1:-1, :]
            log_p_ref_all = F.log_softmax(logits_ref, dim=-1)
            ref_log_probs = log_p_ref_all.gather(dim=-1, index=targets).squeeze(-1)

        # 6. Compute GRPO Loss & Unbiased KL
        ratios = torch.exp(curr_log_probs - old_log_probs)
        adv_exp = advantages.unsqueeze(-1)
        surr1 = ratios * adv_exp
        surr2 = torch.clamp(ratios, 0.8, 1.2) * adv_exp
        policy_loss = -torch.min(surr1, surr2)
        
        # Unbiased KL
        log_ratio_kl = ref_log_probs - curr_log_probs
        kl_div = torch.exp(log_ratio_kl) - log_ratio_kl - 1.0
        
        total_loss = (policy_loss + self.beta_kl * kl_div).mean()

        # 7. Gradient Step
        self.optimizer.zero_grad()
        total_loss.backward()
        nn.utils.clip_grad_norm_(self.policy.parameters(), max_norm=1.0)
        self.optimizer.step()

        return rewards_t.mean().item(), total_loss.item()


# =====================================================================
# 4. Active Pipeline Execution
# =====================================================================

if __name__ == "__main__":
    tokenizer = R1Tokenizer()
    
    policy = CausalLLM(vocab_size=tokenizer.vocab_size)
    ref_policy = CausalLLM(vocab_size=tokenizer.vocab_size)
    ref_policy.load_state_dict(policy.state_dict())
    
    # Freeze reference policy
    for param in ref_policy.parameters():
        param.requires_grad = False

    trainer = FullGRPOTrainer(
        policy=policy, 
        ref_policy=ref_policy, 
        tokenizer=tokenizer,
        group_size=4,
        ref_update_steps=3 # Update reference policy every 3 steps for demo
    )

    prompt = "10*2-5="
    truth = "15"

    print("=== Executing DeepSeek-R1-Zero Pure RL Loop ===")
    for step in range(1, 7):
        mean_r, loss_val = trainer.step(prompt, truth, step_idx=step)
        print(f"Step {step:02d} | Avg Group Reward: {mean_r:.2f} | GRPO Loss: {loss_val:.6f}")

=== Executing DeepSeek-R1-Zero Pure RL Loop ===
Step 01 | Avg Group Reward: 0.00 | GRPO Loss: 0.000000
Step 02 | Avg Group Reward: 0.00 | GRPO Loss: 0.000000
  [Pipeline] Step 3: Updating Reference Policy (pi_ref <- pi_theta)...
Step 03 | Avg Group Reward: 0.00 | GRPO Loss: 0.000000
Step 04 | Avg Group Reward: 0.00 | GRPO Loss: 0.000000
Step 05 | Avg Group Reward: 0.00 | GRPO Loss: 0.000000
  [Pipeline] Step 6: Updating Reference Policy (pi_ref <- pi_theta)...
Step 06 | Avg Group Reward: 0.00 | GRPO Loss: 0.000000
